# Find the best PDB structure for a gene

Enter a gene name, retrieve the reference protein sequence from UniProt and search RCSB PDB for the best matching structures to visualize.

In [11]:
from pathlib import Path

import pandas as pd
import requests
from Bio.PDB import PDBParser
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "TCRTools").exists() and not (NOTEBOOK_DIR / "data").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "TCRTools"

OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

UNIPROT_SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"
RCSB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_DATA_URL = "https://data.rcsb.org/rest/v1/core"

In [12]:
GENE_NAME = "TRAV10"
ORGANISM_ID = 9606
REVIEWED_UNIPROT_ONLY = True
UNIPROT_ROWS = 5
PDB_ROWS = 20
IDENTITY_CUTOFF = 0.30
EVALUE_CUTOFF = 1

In [13]:
def clean_sequence(sequence):
    return "".join(str(sequence).upper().replace("-", "").split())


def extract_uniprot_protein_name(result):
    description = result.get("proteinDescription", {})
    recommended = description.get("recommendedName", {})
    if recommended:
        return recommended.get("fullName", {}).get("value", "")
    submissions = description.get("submissionNames") or []
    if submissions:
        return submissions[0].get("fullName", {}).get("value", "")
    return ""


def extract_uniprot_gene_names(result):
    names = []
    for gene in result.get("genes", []) or []:
        gene_name = gene.get("geneName", {}).get("value")
        if gene_name:
            names.append(gene_name)
        for synonym in gene.get("synonyms", []) or []:
            synonym_name = synonym.get("value")
            if synonym_name:
                names.append(synonym_name)
    return ",".join(dict.fromkeys(names))


def resolve_uniprot_by_gene(gene_name, organism_id=9606, reviewed=True, rows=5):
    query_parts = [f"gene_exact:{gene_name}", f"organism_id:{organism_id}"]
    if reviewed:
        query_parts.append("reviewed:true")
    response = requests.get(
        UNIPROT_SEARCH_URL,
        params={
            "query": " AND ".join(query_parts),
            "format": "json",
            "fields": "accession,id,protein_name,gene_names,organism_name,sequence,length",
            "size": rows,
        },
        timeout=30,
    )
    response.raise_for_status()
    records = []
    for rank, result in enumerate(response.json().get("results", []), start=1):
        sequence = result.get("sequence", {})
        organism = result.get("organism", {})
        records.append(
            {
                "rank": rank,
                "gene_query": gene_name,
                "accession": result.get("primaryAccession"),
                "uniprot_id": result.get("uniProtkbId"),
                "protein_name": extract_uniprot_protein_name(result),
                "gene_names": extract_uniprot_gene_names(result),
                "organism": organism.get("scientificName"),
                "length": sequence.get("length"),
                "sequence": clean_sequence(sequence.get("value", "")),
            }
        )
    return pd.DataFrame(records)


def first_match_context(hit):
    for service in hit.get("services", []):
        for node in service.get("nodes", []):
            contexts = node.get("match_context") or []
            if contexts:
                context = contexts[0].copy()
                context["bitscore"] = context.get("bitscore", node.get("original_score"))
                return context
    return {}


def rcsb_sequence_search(sequence, rows=10, identity_cutoff=0.3, evalue_cutoff=1):
    payload = {
        "query": {
            "type": "terminal",
            "service": "sequence",
            "parameters": {
                "evalue_cutoff": evalue_cutoff,
                "identity_cutoff": identity_cutoff,
                "target": "pdb_protein_sequence",
                "value": clean_sequence(sequence),
            },
        },
        "return_type": "polymer_entity",
        "request_options": {
            "paginate": {"start": 0, "rows": rows},
            "results_verbosity": "verbose",
            "scoring_strategy": "sequence",
        },
    }
    response = requests.post(RCSB_SEARCH_URL, json=payload, timeout=30)
    if response.status_code == 204:
        return []
    response.raise_for_status()
    hits = []
    for rank, hit in enumerate(response.json().get("result_set", []), start=1):
        entry_id, entity_id = hit["identifier"].split("_", 1)
        hits.append(
            {
                "rank": rank,
                "identifier": hit["identifier"],
                "entry_id": entry_id,
                "entity_id": entity_id,
                "score": hit.get("score"),
                **first_match_context(hit),
            }
        )
    return hits


def polymer_entity_metadata(entry_id, entity_id):
    response = requests.get(f"{RCSB_DATA_URL}/polymer_entity/{entry_id}/{entity_id}", timeout=30)
    response.raise_for_status()
    data = response.json()
    identifiers = data.get("rcsb_polymer_entity_container_identifiers", {})
    entity = data.get("rcsb_polymer_entity", {})
    return {
        "description": entity.get("pdbx_description"),
        "asym_ids": ",".join(identifiers.get("asym_ids", [])),
        "auth_asym_ids": ",".join(identifiers.get("auth_asym_ids", [])),
    }


def entry_metadata(entry_id):
    response = requests.get(f"{RCSB_DATA_URL}/entry/{entry_id}", timeout=30)
    response.raise_for_status()
    data = response.json()
    methods = [item.get("method") for item in data.get("exptl", []) if item.get("method")]
    resolutions = [item.get("ls_d_res_high") for item in data.get("refine", []) if item.get("ls_d_res_high") is not None]
    resolutions += [item.get("resolution") for item in data.get("em_3d_reconstruction", []) if item.get("resolution") is not None]
    return {
        "experimental_method": "; ".join(methods),
        "resolution": min(resolutions) if resolutions else None,
    }


def rank_structure_hits(hits):
    ranked = hits.copy()
    for column in ["sequence_identity", "query_coverage", "resolution", "score", "evalue"]:
        if column in ranked.columns:
            ranked[column] = pd.to_numeric(ranked[column], errors="coerce")
    ranked = ranked.sort_values(
        ["sequence_identity", "query_coverage", "resolution", "score", "evalue", "rank"],
        ascending=[False, False, True, False, True, True],
        na_position="last",
    ).reset_index(drop=True)
    ranked.insert(0, "visualization_rank", range(1, len(ranked) + 1))
    return ranked


def search_best_structures_for_sequence(sequence, query_name="query", rows=10, identity_cutoff=0.3, evalue_cutoff=1):
    records = []
    for hit in rcsb_sequence_search(sequence, rows=rows, identity_cutoff=identity_cutoff, evalue_cutoff=evalue_cutoff):
        query_length = hit.get("query_length") or len(clean_sequence(sequence))
        alignment_length = hit.get("alignment_length")
        query_coverage = alignment_length / query_length if query_length and alignment_length else None
        records.append(
            {
                "query_name": query_name,
                **hit,
                "query_coverage": query_coverage,
                **polymer_entity_metadata(hit["entry_id"], hit["entity_id"]),
                **entry_metadata(hit["entry_id"]),
            }
        )
    return rank_structure_hits(pd.DataFrame(records)) if records else pd.DataFrame()


def download_pdb(pdb_id, output_dir):
    output_path = Path(output_dir) / f"{str(pdb_id).upper()}.pdb"
    if not output_path.exists():
        response = requests.get(f"https://files.rcsb.org/download/{str(pdb_id).upper()}.pdb", timeout=30)
        response.raise_for_status()
        output_path.write_text(response.text)
    return output_path


def first_auth_chain(auth_asym_ids):
    return str(auth_asym_ids).split(",")[0].strip()


def summarize_pdb_chains(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(pdb_path).stem, pdb_path)
    rows = []
    for chain in structure[0]:
        residues = [residue for residue in chain if residue.id[0] == " "]
        atoms = sum(1 for residue in residues for _atom in residue)
        rows.append({"chain": chain.id, "atoms": atoms, "residues": len(residues)})
    return pd.DataFrame(rows)


def chain_label_positions(pdb_path, chain_ids=None):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(pdb_path).stem, pdb_path)
    requested = set(chain_ids) if chain_ids is not None else None
    labels = []
    for chain in structure[0]:
        if requested is not None and chain.id not in requested:
            continue
        residues = [residue for residue in chain if residue.id[0] == " "]
        if not residues:
            continue
        residue = residues[len(residues) // 2]
        labels.append({"chain": chain.id, "residue_number": str(residue.id[1])})
    return labels


def add_chain_labels(view, pdb_path, chain_ids=None):
    for label in chain_label_positions(pdb_path, chain_ids=chain_ids):
        selection = f":{label['chain']} and {label['residue_number']} and .CA"
        view.add_label(
            selection=selection,
            labelType="text",
            labelText=f"chain {label['chain']}",
            color="black",
            labelSize=2.0,
            zOffset=2.0,
        )
    return view


def ngl_selection(chain_id):
    return f":{chain_id}"

In [14]:
uniprot_candidates = resolve_uniprot_by_gene(
    GENE_NAME,
    organism_id=ORGANISM_ID,
    reviewed=REVIEWED_UNIPROT_ONLY,
    rows=UNIPROT_ROWS,
)
if uniprot_candidates.empty:
    raise ValueError(f"No UniProt candidates found for gene {GENE_NAME}")

uniprot_candidates.to_csv(OUTPUT_DIR / f"{GENE_NAME}_uniprot_candidates.csv", index=False)
uniprot_candidates[["rank", "accession", "uniprot_id", "protein_name", "gene_names", "organism", "length"]]

,rank,accession,uniprot_id,protein_name,gene_names,organism,length
0,1,A0A0B4J240,TVA10_HUMAN,T cell receptor alpha variable 10,TRAV10,Homo sapiens,114


In [15]:
selected_protein = uniprot_candidates.iloc[0]
selected_protein[["accession", "uniprot_id", "protein_name", "length"]]

accession                              A0A0B4J240
uniprot_id                            TVA10_HUMAN
protein_name    T cell receptor alpha variable 10
length                                        114
Name: 0, dtype: object

In [16]:
structure_hits = search_best_structures_for_sequence(
    selected_protein["sequence"],
    query_name=selected_protein["accession"],
    rows=PDB_ROWS,
    identity_cutoff=IDENTITY_CUTOFF,
    evalue_cutoff=EVALUE_CUTOFF,
)
if structure_hits.empty:
    raise ValueError(f"No PDB structure hits found for gene {GENE_NAME}")

structure_hits.insert(0, "gene_query", GENE_NAME)
structure_hits.insert(1, "uniprot_accession", selected_protein["accession"])
structure_hits.insert(2, "protein_name", selected_protein["protein_name"])
structure_hits.to_csv(OUTPUT_DIR / f"{GENE_NAME}_pdb_search_results.csv", index=False)

columns = [
    "visualization_rank", "identifier", "description", "auth_asym_ids", "sequence_identity",
    "query_coverage", "evalue", "bitscore", "experimental_method", "resolution",
]
structure_hits[columns].head(10)

,visualization_rank,identifier,description,auth_asym_ids,sequence_identity,query_coverage,evalue,bitscore,experimental_method,resolution
0,1,3TYF_1,iNKT Cell Receptor Alpha Chain,"A,C",1.0,0.824561,1.431000e-55,190,X-RAY DIFFRACTION,2.806
1,2,3TZV_1,Invariant Natural Killer T Cell Receptor chain A,"A,G",1.0,0.824561,1.431000e-55,190,X-RAY DIFFRACTION,3.056
2,3,2EYS_1,NKT15,A,1.0,0.815789,9.531000e-55,188,X-RAY DIFFRACTION,2.210
3,4,2EYR_1,NKT12,A,1.0,0.815789,9.531000e-55,188,X-RAY DIFFRACTION,2.400
4,5,3HUJ_3,NKT15 T cell receptor alpha-chain,"E,G",1.0,0.815789,9.531000e-55,188,X-RAY DIFFRACTION,2.500
5,6,2EYT_1,NKT15,"A,C",1.0,0.815789,9.531000e-55,188,X-RAY DIFFRACTION,2.600
6,7,3VWK_3,NKT15 T cell receptor alpha-chain,C,1.0,0.815789,9.531000e-55,188,X-RAY DIFFRACTION,2.935
7,8,3VWJ_3,NKT15 T cell receptor alpha-chain,C,1.0,0.815789,9.531000e-55,188,X-RAY DIFFRACTION,3.093
8,9,6V80_3,"T cell receptor alpha variable 10, nkt tcr alp...","C,H",1.0,0.815789,9.531000e-55,188,X-RAY DIFFRACTION,3.530
9,10,3SDX_3,NKT TCR Valpha24 chain,"E,G",1.0,0.807018,4.629000e-54,186,X-RAY DIFFRACTION,3.120


In [17]:
best_hit = structure_hits.iloc[0]
best_hit.rename("value").to_frame()

,value
gene_query,TRAV10
uniprot_accession,A0A0B4J240
protein_name,T cell receptor alpha variable 10
visualization_rank,1
query_name,A0A0B4J240
rank,1
identifier,3TYF_1
entry_id,3TYF
entity_id,1
score,1.0


In [18]:
structure_path = download_pdb(best_hit["entry_id"], OUTPUT_DIR)
display_chain = first_auth_chain(best_hit["auth_asym_ids"])

pd.DataFrame(
    [
        {
            "gene_query": GENE_NAME,
            "uniprot_accession": selected_protein["accession"],
            "entry_id": best_hit["entry_id"],
            "entity_id": best_hit["entity_id"],
            "display_chain": display_chain,
            "structure_path": structure_path.as_posix(),
        }
    ]
).to_csv(OUTPUT_DIR / f"{GENE_NAME}_best_structure.csv", index=False)

structure_path, display_chain

(PosixPath('/home/jordivilla/GitHub/RESEARCH/Tools/TCRTools/output/3TYF.pdb'),
 'A')

In [19]:
summarize_pdb_chains(structure_path)

,chain,atoms,residues
0,A,1543,199
1,B,1893,240
2,C,1511,197
3,D,1915,242


In [20]:
import nglview as nv

view = nv.show_structure_file(str(structure_path))
view.clear_representations()
view.add_cartoon(selection="protein", color="lightgray")
view.add_cartoon(selection=ngl_selection(display_chain), color="royalblue")
add_chain_labels(view, structure_path)
view.center(ngl_selection(display_chain))
view

NGLWidget()